# 预处理文本数据

In [1]:
from pathlib import Path
import pandas as pd
import re
import sqlite3


## 配置路径

In [2]:
# 配置数据集基本路径
folderpath_original_data_root = Path.cwd().parent.parent.parent.joinpath('EntelechyData/original')
Path.mkdir(folderpath_original_data_root, exist_ok=True)

folderpath_pretreat_data_root = Path.cwd().parent.parent.parent.joinpath('EntelechyData/pretreatment')
Path.mkdir(folderpath_pretreat_data_root, exist_ok=True)

# 配置数据集路径
folderpath_pretreat_现代汉语词典 = Path(folderpath_pretreat_data_root, f'现代汉语词典')
folderpath_pretreat_新华字典 = Path(folderpath_pretreat_data_root, f'新华字典')
folderpath_pretreat_现代汉语常用字符集 = Path(folderpath_pretreat_data_root, f'现代汉语常用字符集')


## 导入预处理的词库数据

### 导入 sqlite 数据库

In [3]:
# 导入 sqlite 数据库
conn_现代汉语词典 = sqlite3.connect(Path(folderpath_pretreat_现代汉语词典, '现代汉语词典.db'))
conn_新华字典 = sqlite3.connect(Path(folderpath_pretreat_新华字典, '新华字典.db'))


### 导入 pandas 数据

In [4]:
# 读取新华字典 pandas 数据库内容
df_新华字典 = pd.read_pickle(Path(folderpath_pretreat_新华字典, '新华字典.pkl'))
# 读取现代汉语词典 pandas 数据库内容
df_现代汉语词典 = pd.read_pickle(Path(folderpath_pretreat_现代汉语词典, '现代汉语词典.pkl'))
df_现代汉语常用字符集 = pd.read_pickle(Path(folderpath_pretreat_现代汉语常用字符集, '现代汉语常用字符集.pkl'))

## 导入到自定义的初始的基础概念库 

### 初始化自定义的初始的基础概念库

In [15]:
# 初始化自定义的初始的基础概念库
df_基础概念_现代汉语字符库 = pd.DataFrame(
    columns=['uid', 'id_概念', 'name_概念', '内容_010', '基础类别', '是否常用字', '备注'])
df_基础概念_现代汉语词库 = pd.DataFrame(columns=['uid', 'id_概念', 'name_概念', '内容_010', '是否常用词', '备注'])

df_基础概念_现代汉语字符库['内容_010'] = df_新华字典['word']  # 导入新华字典 pandas 数据库内容到自定义的初始的基础概念库
df_基础概念_现代汉语字符库['基础类别'] = '汉字集'
df_基础概念_现代汉语字符库['是否常用字'] = '否'


### 【df_基础概念_现代汉语字符库】 标记常用字

In [16]:


# 标记常用字
for index, row in df_现代汉语常用字符集.iterrows():
    if row['类别'] == '汉字集':
        if row['字符'] in df_基础概念_现代汉语字符库['内容_010'].values:
            index_02 = df_基础概念_现代汉语字符库[df_基础概念_现代汉语字符库['内容_010'] == row['字符']].index[0]
            df_基础概念_现代汉语字符库.loc[index_02, '是否常用字'] = '是'


In [17]:
df_基础概念_现代汉语字符库_2 = df_基础概念_现代汉语字符库.copy()
# 插入非汉字集的字符
for index, row in df_现代汉语常用字符集.iterrows():
    if row['类别'] != '汉字集':
        df_基础概念_现代汉语字符库_2 = pd.concat(
            [
                df_基础概念_现代汉语字符库,
                pd.DataFrame({
                    '内容_010': [row['字符']],
                    '基础类别': [row['类别']],
                    '是否常用字': ['是']
                })
            ]
        )

